# 🎙 ThaiVoiceLab V3 — โคลนเสียงภาษาไทยบน Xiaomi Pad 6 ฟรีค่า API

**ไม่ต้องมี PC ไม่ต้องใช้ ElevenLabs** — ใช้โมเดล Open Source F5-TTS-THAI บนโน้ตบุ๊ก Google Colab

1. เปิดใน Chrome, เชื่อมต่อ Google Colab; ถ้ามี GPU ให้เลือก Runtime → Change runtime type → T4 GPU
2. กด ▶ ที่เซลล์ **1 ติดตั้ง AI**
3. ตั้งค่าในเซลล์ **2** (คำพูดจริงในเสียงต้นแบบ 2–8 วินาที + ข้อความใหม่), ติ๊กยืนยันสิทธิ์เสียง แล้วกด ▶
4. อัปโหลดไฟล์เสียงจากแท็บเล็ต แล้วฟัง/ดาวน์โหลด MP3 หลังสร้างเสร็จ

⚠️ Colab Free **ไม่รับประกัน GPU/เวลาใช้งาน** และอาจจำกัดโควตา CPU จะช้ามาก การใช้งานนี้เป็นการรันโน้ตบุ๊กโดยตรง ไม่เปิด Worker หรือเซิร์ฟเวอร์เชื่อมต่อจากเว็บ

🔒 ใช้เฉพาะเสียงตัวเองหรือเสียงที่ได้รับอนุญาต ไฟล์เสียงที่อัปโหลดส่งไปประมวลผลใน Colab runtime ภายใต้บัญชี Google ของคุณ ไม่ส่งเข้า Render


In [ ]:
# @title 1. ติดตั้ง AI ภาษาไทย
import sys, shutil, subprocess
print("ติดตั้ง F5-TTS ภาษาไทย โดยไม่ใช้ API Key")
if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):
    subprocess.run(["apt-get","-qq","update"],check=True)
    subprocess.run(["apt-get","-qq","install","-y","ffmpeg"],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","f5-tts-th","numpy","soundfile"],check=True)
import torch
print("✅ ติดตั้งแล้ว |", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU — ช้ามาก")


In [ ]:
# @title 2. สร้างเสียงไทยด้วยตัวอย่างเสียงของคุณ
# @markdown คำพูดจริงใน 8 วินาทีแรกของเสียงต้นแบบ
ref_text = "สวัสดีครับ วันนี้อากาศดีมากเลยครับ" # @param {type:"string"}
# @markdown ข้อความใหม่ที่ต้องการให้พูด
new_text = "สวัสดีครับ นี่คือเสียงที่สร้างด้วย ThaiVoiceLab" # @param {type:"string"}
speed = 1.0 # @param {type:"slider", min:0.7, max:1.3, step:0.1}
steps = 24 # @param {type:"slider", min:16, max:40, step:4}
# @markdown ยืนยันว่าเป็นเสียงตนเองหรือได้รับอนุญาต
have_permission = False # @param {type:"boolean"}
import re, subprocess, tempfile, shutil
from pathlib import Path
import numpy as np
import soundfile as sf
from IPython.display import display, Audio
from google.colab import files

if not have_permission: raise ValueError("กรุณาติ๊กยืนยันสิทธิ์การใช้เสียง")
ref_text=ref_text.strip(); new_text=new_text.strip()
if not 1<=len(ref_text)<=300: raise ValueError("ต้องใส่คำพูดจริงในเสียงต้นแบบ 1-300 อักขระ")
if not 1<=len(new_text)<=1500: raise ValueError("ต้องใส่บทพูดใหม่ 1-1500 อักขระ")
if not .7<=speed<=1.3 or not 16<=steps<=40: raise ValueError("ตั้งค่าความเร็วหรือคุณภาพไม่ถูกต้อง")
print("เลือกไฟล์เสียง 2-8 วินาทีของคุณจากแท็บเล็ต")
received=files.upload()
if not received: raise ValueError("ไม่ได้เลือกไฟล์")
filename, audio_bytes=next(iter(received.items()))
ext=Path(filename).suffix.lower()
if ext not in {".mp3",".wav",".m4a",".ogg",".webm",".flac"}: raise ValueError("ไฟล์ไม่รองรับ")
if not 1000<=len(audio_bytes)<=15*1024*1024: raise ValueError("ไฟล์ต้องไม่เกิน 15 MB")
with tempfile.TemporaryDirectory(prefix="thai_voice_") as tmp:
    src=Path(tmp)/("input"+ext); src.write_bytes(audio_bytes)
    result=subprocess.run(["ffprobe","-v","error","-show_entries","format=duration","-of","default=nokey=1:noprint_wrappers=1",str(src)],capture_output=True,text=True,check=True,timeout=30)
    duration=float(result.stdout.strip())
    if not 2<=duration<=180: raise ValueError("ไฟล์ต้นแบบยาวได้ 2-180 วินาที โดยใช้ 8 วินาทีแรก")
    ref=Path(tmp)/"reference.wav"
    subprocess.run(["ffmpeg","-nostdin","-hide_banner","-loglevel","error","-y","-i",str(src),"-vn","-ac","1","-ar","24000","-t","8",str(ref)],check=True,timeout=120)
    if ref.stat().st_size<1000: raise ValueError("ไฟล์ไม่มีเสียง")
    from f5_tts_th.tts import TTS
    if "_thai_voice_model" not in globals():
        print("กำลังโหลดโมเดลเป็นครั้งแรก")
        _thai_voice_model=TTS(model="v1")
    def chunks(text,limit=170):
        text=re.sub(r"\s+"," ",text.strip())
        while text:
            if len(text)<=limit: yield text; break
            matches=[m.end() for m in re.finditer(r"[\s,.;!?。！？ฯๆ]",text[:limit+1]) if m.end()>=limit//2]
            cut=max(matches) if matches else limit
            yield text[:cut].strip(); text=text[cut:].strip()
    segments=list(chunks(new_text))
    audio_parts=[]
    for i,part in enumerate(segments):
        print("สร้างเสียง",i+1,"/",len(segments))
        w=_thai_voice_model.infer(ref_audio=str(ref),ref_text=ref_text,gen_text=part,step=steps,cfg=2.0,speed=speed)
        arr=np.asarray(w,dtype=np.float32).reshape(-1)
        if arr.size==0 or not np.isfinite(arr).all(): raise RuntimeError("โมเดลไม่ส่งเสียงกลับมา")
        audio_parts.append(arr)
        if i<len(segments)-1:audio_parts.append(np.zeros(3840,dtype=np.float32))
    wav=Path(tmp)/"out.wav"
    sf.write(str(wav),np.clip(np.concatenate(audio_parts),-1,1),24000,subtype="PCM_16")
    out=str(Path.cwd()/"ThaiVoiceLab_Thai_Clone.mp3")
    subprocess.run(["ffmpeg","-nostdin","-hide_banner","-loglevel","error","-y","-i",str(wav),"-c:a","libmp3lame","-b:a","128k",out],check=True,timeout=180)
    if Path(out).stat().st_size<1000: raise RuntimeError("ไม่สามารถสร้าง MP3")
print("✅ สร้างเสียง MP3 สำเร็จแล้ว")
display(Audio(filename=out))
files.download(out)


โมเดล: https://github.com/VYNCX/F5-TTS-THAI และ https://huggingface.co/VIZINTZOR/F5-TTS-THAI
ข้อจำกัด Colab: https://research.google.com/colaboratory/faq.html